# Notebook 5 — Deteksi Jenis Kendaraan & Bounding Box dengan SSD MobileNet V2 dari Output CSV/JSON Notebook 1

Notebook Google Colab ini menghubungkan langsung output **Notebook 1 (`1_pcv_klasik.ipynb`)** ke model **SSD MobileNet V2 FPNLite 320×320** milikmu (`pcv-model-colab-v4-ckpt-42.zip` / `SavedModel`) sehingga kamu cukup **mengunggah file CSV atau JSON hasil Notebook 1** untuk langsung mendapatkan:
- **Jenis Kendaraan (`Mobil`, `Truk`, `Bus`)** + **Confidence Score**,
- **Bounding Box SSD MobileNet V2** (piksel & ternormalisasi `0..1`),
- **Perbandingan langsung (IoU & visual)** antara Bounding Box PCV Klasik (Notebook 1, tanpa label) vs Bounding Box + Label hasil SSD MobileNet V2,
- **File ekspor lengkap** (`hasil_deteksi_ssd_dari_nb1.csv`, `hasil_deteksi_ssd_dari_nb1.json`, gambar ber-bbox & JSON per gambar, serta ZIP siap unduh).

### Format Input Notebook 1 yang Didukung Otomatis
1. **CSV gabungan**: `07_ringkasan_aturan_if_else.csv`
2. **JSON gabungan**: `07_hasil_bounding_box_semua.json`
3. **JSON per gambar**: satu atau banyak file `per_gambar/0001_<nama>.json`
4. **ZIP lengkap Notebook 1**: `nb1_pcv_klasik_outputs.zip`

### Catatan Teknis Penting
* Model **SSD MobileNet V2** tetap memerlukan piksel gambar RGB (bukan hanya angka koordinat). Notebook ini membaca `File_Name` / `file_name` dan koordinat BBox dari CSV/JSON Notebook 1, lalu mencocokkannya ke folder gambar asli (default otomatis di Google Drive `MyDrive/ferqueue/gambar`, sama persis dengan Notebook 1 — atau bisa lewat upload gambar).
* Tidak perlu instalasi TF Object Detection API / protobuf / venv: langsung memakai **TensorFlow bawaan Google Colab** untuk memuat `SavedModel`!
* Mendukung 2 mode inferensi di Cell 5:
  - `INFERENCE_MODE = "full_image"` *(★ Default / Rekomendasi)*: SSD membaca gambar utuh sesuai kondisi training (akurasi tertinggi), lalu menyandingkan & mengisi label pada data CSV/JSON Notebook 1.
  - `INFERENCE_MODE = "roi_crop"`: SSD memotong gambar memakai area BBox dari CSV/JSON Notebook 1 (+ padding aman 15%), mendeteksi di dalam ROI, lalu memetakan koordinat BBox kembali ke gambar asli (dengan *fallback* otomatis ke gambar utuh bila ROI terlalu sempit).

## Cell 1 — Setup Library & Folder Output

In [ ]:
import json
import math
import random
import shutil
import time
import zipfile
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # type: ignore[name-defined]
except NameError:
    display = print

OUT_DIR = Path("/content/outputs_presentasi/nb5_ssd_dari_nb1") if IN_COLAB else Path("./outputs_presentasi/nb5_ssd_dari_nb1")
MODEL_WORK_DIR = Path("/content/ferqueue_ssd_model") if IN_COLAB else Path("./outputs_presentasi/_model_cache_nb5")
INPUT_WORK_DIR = OUT_DIR / "_nb1_inputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_WORK_DIR.mkdir(parents=True, exist_ok=True)
INPUT_WORK_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"


def save_fig(fig, filename: str) -> Path:
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Tersimpan: {path}")
    return path


def ensure_drive_mounted() -> bool:
    if not IN_COLAB:
        return False
    if Path("/content/drive/MyDrive").is_dir():
        return True
    drive.mount("/content/drive")
    return Path("/content/drive/MyDrive").is_dir()


print(
    f"Siap! Colab={IN_COLAB} | TensorFlow={tf.__version__} | OpenCV={cv2.__version__} | "
    f"GPU={len(tf.config.list_physical_devices('GPU')) > 0} | Output={OUT_DIR}"
)

## Cell 2 — Muat Model SSD MobileNet V2 FERQUEUE (`pcv-model-colab-v4-ckpt-42.zip`)

Pilihan `MODEL_SOURCE`:
* `"auto"` *(★ Default)*: Di Colab otomatis mencari model terbaru (`v4` / `v3`) di Google Drive; jika tidak ditemukan, langsung membuka dialog upload file ZIP model (`pcv-model-colab-v4-ckpt-42.zip`). Di lokal otomatis memakai `models/mobilenet/runs/pcv-receiver-v4/...` atau `outputs/pcv-model-colab-v4-ckpt-42.zip`.
* `"upload_zip"`: Selalu unggah file ZIP model langsung dari laptop.
* `"drive"`: Muat langsung dari `DRIVE_MODEL_PATH`.
* `"local"`: Muat dari path lokal repo FERQUEUE.

In [ ]:
MODEL_SOURCE = "auto"  # Pilihan: "auto" | "upload_zip" | "drive" | "local"
DRIVE_MODEL_PATH = "/content/drive/MyDrive/ferqueue/pcv-coco-colab-v4/pcv-model-colab-v4-ckpt-42.zip"
DRIVE_MODEL_FALLBACKS = [
    "/content/drive/MyDrive/ferqueue/pcv-model-colab-v4-ckpt-42.zip",
    "/content/drive/MyDrive/ferqueue/pcv-coco-colab-v3/pcv-model-colab-v3-ckpt-35.zip",
]
LOCAL_MODEL_CANDIDATES = [
    "models/mobilenet/runs/pcv-receiver-v4/exports/imported-ckpt-42",
    "outputs/pcv-model-colab-v4-ckpt-42.zip",
    "models/mobilenet/runs/pcv-receiver-v3-unfrozen/exports/imported-ckpt-35",
    "outputs/pcv-model-colab-v3-ckpt-35.zip",
]

DEFAULT_CLASSES = ["Mobil", "Truk", "Bus"]


def resolve_saved_model(src_mode: str):
    extract_dir = MODEL_WORK_DIR / "extracted"
    meta = {
        "classes": DEFAULT_CLASSES,
        "model": "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8",
        "checkpoint": "unknown",
        "preprocessing": {"denoise_kernel": 0, "clahe_clip_limit": 0, "clahe_grid_size": 8, "gamma": 1.0},
    }

    target_path = None
    if src_mode == "upload_zip" and IN_COLAB:
        print("Silakan unggah file ZIP model SSD MobileNet V2 (mis. pcv-model-colab-v4-ckpt-42.zip)...")
        uploaded = colab_files.upload()
        zip_name, content = next(iter(uploaded.items()))
        target_path = MODEL_WORK_DIR / Path(zip_name).name
        target_path.write_bytes(content)
    elif src_mode in ("drive", "auto") and IN_COLAB:
        if ensure_drive_mounted():
            for cand in [DRIVE_MODEL_PATH, *DRIVE_MODEL_FALLBACKS]:
                p = Path(cand)
                if p.exists():
                    target_path = p
                    print(f"Model ditemukan di Google Drive: {target_path}")
                    break
        if target_path is None:
            if src_mode == "drive":
                raise FileNotFoundError(f"Model tidak ditemukan di Google Drive: {DRIVE_MODEL_PATH}")
            print("Model belum ditemukan di Google Drive. Silakan unggah file ZIP model (mis. pcv-model-colab-v4-ckpt-42.zip)...")
            uploaded = colab_files.upload()
            zip_name, content = next(iter(uploaded.items()))
            target_path = MODEL_WORK_DIR / Path(zip_name).name
            target_path.write_bytes(content)
    else:
        for cand in LOCAL_MODEL_CANDIDATES:
            p = Path(cand)
            if p.exists():
                target_path = p
                break
        if target_path is None:
            raise FileNotFoundError(
                "Model lokal tidak ditemukan. Pastikan menjalankan dari root repo FERQUEUE atau ubah LOCAL_MODEL_CANDIDATES."
            )

    if target_path.is_file() and target_path.suffix.lower() == ".zip":
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target_path) as zf:
            zf.extractall(extract_dir)
        if (extract_dir / "model.json").is_file():
            meta.update(json.loads((extract_dir / "model.json").read_text(encoding="utf-8")))
        saved_dir = extract_dir / "saved_model"
    elif target_path.is_dir():
        saved_dir = target_path / "saved_model" if (target_path / "saved_model/saved_model.pb").is_file() else target_path
        for export_json in (target_path / "model.json", target_path.parent.parent / "export.json"):
            if export_json.is_file():
                meta.update(json.loads(export_json.read_text(encoding="utf-8")))
                break
    else:
        raise FileNotFoundError(f"Path model tidak valid: {target_path}")

    if not (saved_dir / "saved_model.pb").is_file():
        raise FileNotFoundError(f"saved_model.pb tidak ditemukan di dalam: {saved_dir}")
    return saved_dir, meta


saved_model_dir, model_meta = resolve_saved_model(MODEL_SOURCE)
CLASSES = list(model_meta["classes"])
MODEL_PREPROCESSING = dict(model_meta.get("preprocessing", {}))

print(f"Memuat SavedModel dari: {saved_model_dir} ...")
t0 = perf_counter()
loaded_tf_model = tf.saved_model.load(str(saved_model_dir))
detector_fn = loaded_tf_model.signatures["serving_default"]
print(f"Model siap dalam {perf_counter() - t0:.2f} detik!")
print(f"Arsitektur : {model_meta['model']} | Checkpoint: {model_meta['checkpoint']}")
print(f"Kelas      : {CLASSES}")
print(f"Preprocess : {MODEL_PREPROCESSING}")

## Cell 3 — Upload / Muat File CSV atau JSON Output dari Notebook 1 (`1_pcv_klasik.ipynb`)

Default di Colab adalah **`NB1_INPUT_SOURCE = "upload"`** — tinggal jalankan cell ini lalu pilih salah satu file output dari Notebook 1:
* `07_ringkasan_aturan_if_else.csv` *(CSV gabungan Notebook 1)*, **atau**
* `07_hasil_bounding_box_semua.json` *(JSON gabungan Notebook 1)*, **atau**
* Satu/banyak file `per_gambar/0001_<nama>.json` *(JSON per gambar)*, **atau**
* `nb1_pcv_klasik_outputs.zip` *(ZIP paket lengkap output Notebook 1)*.

> Jika file output Notebook 1 sudah tersimpan di Google Drive (`MyDrive/ferqueue/output_pcv_klasik/`), kamu juga bisa mengganti `NB1_INPUT_SOURCE = "drive"` agar tidak perlu upload manual.

In [ ]:
NB1_INPUT_SOURCE = "upload"  # Pilihan: "upload" | "drive" | "local"
DRIVE_NB1_INPUT_PATH = "/content/drive/MyDrive/ferqueue/output_pcv_klasik/07_hasil_bounding_box_semua.json"
DRIVE_NB1_FALLBACKS = [
    "/content/drive/MyDrive/ferqueue/output_pcv_klasik/07_ringkasan_aturan_if_else.csv",
    "/content/drive/MyDrive/ferqueue/output_pcv_klasik/nb1_pcv_klasik_outputs.zip",
]
LOCAL_NB1_INPUT_PATH = "outputs_presentasi/nb1_pcv_klasik/07_hasil_bounding_box_semua.json"


def _safe_int(val, default=0) -> int:
    try:
        if val is None or (isinstance(val, float) and math.isnan(val)):
            return default
        return int(round(float(val)))
    except (TypeError, ValueError):
        return default


def _safe_float(val, default=0.0) -> float:
    try:
        if val is None or (isinstance(val, float) and math.isnan(val)):
            return default
        return float(val)
    except (TypeError, ValueError):
        return default


def _normalize_single_json_record(rec: dict, fallback_idx: int) -> dict:
    file_name = str(rec.get("file_name") or rec.get("File_Name") or rec.get("filename") or "").strip()
    img_w = _safe_int(rec.get("image_width") or rec.get("Image_W"), 0)
    img_h = _safe_int(rec.get("image_height") or rec.get("Image_H"), 0)
    objects = rec.get("objects") or []

    nb1_bbox_xyxy = None
    aspect_ratio = 0.0
    extent = 0.0
    solidity = 0.0
    circularity = 0.0

    if isinstance(objects, list) and objects:
        primary = objects[0]
        if isinstance(primary.get("bbox_xyxy_px"), (list, tuple)) and len(primary["bbox_xyxy_px"]) == 4:
            x1, y1, x2, y2 = [_safe_int(v) for v in primary["bbox_xyxy_px"]]
            if x2 > x1 and y2 > y1:
                nb1_bbox_xyxy = [x1, y1, x2, y2]
        elif isinstance(primary.get("bbox_xywh_px"), (list, tuple)) and len(primary["bbox_xywh_px"]) == 4:
            x, y, w, h = [_safe_int(v) for v in primary["bbox_xywh_px"]]
            if w > 0 and h > 0:
                nb1_bbox_xyxy = [x, y, x + w, y + h]
        aspect_ratio = _safe_float(primary.get("aspect_ratio"), 0.0)
        extent = _safe_float(primary.get("extent"), 0.0)
        solidity = _safe_float(primary.get("solidity"), 0.0)
        circularity = _safe_float(primary.get("circularity"), 0.0)
    elif all(k in rec for k in ("BBox_xmin", "BBox_ymin", "BBox_xmax", "BBox_ymax")):
        x1 = _safe_int(rec.get("BBox_xmin"))
        y1 = _safe_int(rec.get("BBox_ymin"))
        x2 = _safe_int(rec.get("BBox_xmax"))
        y2 = _safe_int(rec.get("BBox_ymax"))
        if x2 > x1 and y2 > y1:
            nb1_bbox_xyxy = [x1, y1, x2, y2]

    stats = rec.get("image_stats") if isinstance(rec.get("image_stats"), dict) else {}
    return {
        "index": _safe_int(rec.get("index") or rec.get("No"), fallback_idx),
        "file_name": file_name,
        "image_width": img_w,
        "image_height": img_h,
        "letterbox": str(stats.get("letterbox", rec.get("Letterbox", ""))),
        "bright_ratio_pct": _safe_float(stats.get("bright_ratio_%", rec.get("Silau_%", 0.0))),
        "grabcut_strategy": str(rec.get("grabcut_strategy") or rec.get("Strategi_if_else") or ""),
        "nb1_object_count": _safe_int(rec.get("object_count", len(objects) if objects else (1 if nb1_bbox_xyxy else 0))),
        "nb1_bbox_xyxy_px": nb1_bbox_xyxy,
        "nb1_aspect_ratio": aspect_ratio or _safe_float(rec.get("Aspect_Ratio"), 0.0),
        "nb1_extent": extent or _safe_float(rec.get("Extent"), 0.0),
        "nb1_solidity": solidity or _safe_float(rec.get("Solidity"), 0.0),
        "nb1_circularity": circularity or _safe_float(rec.get("Circularity"), 0.0),
        "nb1_mask_coverage_pct": _safe_float(rec.get("mask_coverage_pct", rec.get("Cakupan_%", 0.0))),
        "nb1_bbox_image_file": str(rec.get("bbox_image_file") or rec.get("Output_BBox_Image") or ""),
        "nb1_raw_record": rec,
    }


def parse_nb1_csv(csv_path: Path) -> list:
    df = pd.read_csv(csv_path)
    cols_lower = {c.lower(): c for c in df.columns}
    fname_col = cols_lower.get("file_name") or cols_lower.get("filename") or cols_lower.get("image") or cols_lower.get("nama_file")
    if not fname_col:
        raise ValueError(f"Kolom nama file ('File_Name'/'file_name') tidak ditemukan di CSV: {list(df.columns)}")

    records = []
    for i, row in df.iterrows():
        row_dict = row.to_dict()
        fname = str(row_dict.get(fname_col, "")).strip()
        if not fname or fname.lower() == "nan":
            continue

        x1 = _safe_int(row_dict.get(cols_lower.get("bbox_xmin", "BBox_xmin"), row_dict.get(cols_lower.get("xmin", "xmin"), 0)))
        y1 = _safe_int(row_dict.get(cols_lower.get("bbox_ymin", "BBox_ymin"), row_dict.get(cols_lower.get("ymin", "ymin"), 0)))
        x2 = _safe_int(row_dict.get(cols_lower.get("bbox_xmax", "BBox_xmax"), row_dict.get(cols_lower.get("xmax", "xmax"), 0)))
        y2 = _safe_int(row_dict.get(cols_lower.get("bbox_ymax", "BBox_ymax"), row_dict.get(cols_lower.get("ymax", "ymax"), 0)))
        bw = _safe_int(row_dict.get(cols_lower.get("bbox_w", "BBox_w"), 0))
        bh = _safe_int(row_dict.get(cols_lower.get("bbox_h", "BBox_h"), 0))
        if x2 <= x1 and bw > 0:
            x2 = x1 + bw
        if y2 <= y1 and bh > 0:
            y2 = y1 + bh
        nb1_bbox = [x1, y1, x2, y2] if (x2 > x1 and y2 > y1) else None

        records.append({
            "index": _safe_int(row_dict.get(cols_lower.get("no", "No"), i + 1), i + 1),
            "file_name": Path(fname).name,
            "image_width": _safe_int(row_dict.get(cols_lower.get("image_w", "Image_W"), 0)),
            "image_height": _safe_int(row_dict.get(cols_lower.get("image_h", "Image_H"), 0)),
            "letterbox": str(row_dict.get(cols_lower.get("letterbox", "Letterbox"), "")),
            "bright_ratio_pct": _safe_float(row_dict.get(cols_lower.get("silau_%", "Silau_%"), 0.0)),
            "grabcut_strategy": str(row_dict.get(cols_lower.get("strategi_if_else", "Strategi_if_else"), "")),
            "nb1_object_count": _safe_int(row_dict.get(cols_lower.get("jumlah_objek", "Jumlah_Objek"), 1 if nb1_bbox else 0)),
            "nb1_bbox_xyxy_px": nb1_bbox,
            "nb1_aspect_ratio": _safe_float(row_dict.get(cols_lower.get("aspect_ratio", "Aspect_Ratio"), 0.0)),
            "nb1_extent": _safe_float(row_dict.get(cols_lower.get("extent", "Extent"), 0.0)),
            "nb1_solidity": _safe_float(row_dict.get(cols_lower.get("solidity", "Solidity"), 0.0)),
            "nb1_circularity": _safe_float(row_dict.get(cols_lower.get("circularity", "Circularity"), 0.0)),
            "nb1_mask_coverage_pct": _safe_float(row_dict.get(cols_lower.get("cakupan_%", "Cakupan_%"), 0.0)),
            "nb1_bbox_image_file": str(row_dict.get(cols_lower.get("output_bbox_image", "Output_BBox_Image"), "")),
            "nb1_raw_record": {k: (None if pd.isna(v) else v) for k, v in row_dict.items()},
        })
    return records


def parse_nb1_json(json_path: Path) -> list:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    if isinstance(data, dict) and isinstance(data.get("results"), list):
        raw_items = data["results"]
    elif isinstance(data, list):
        raw_items = data
    elif isinstance(data, dict) and any(k in data for k in ("file_name", "File_Name", "filename")):
        raw_items = [data]
    else:
        raise ValueError(f"Struktur JSON tidak dikenali sebagai output Notebook 1: {json_path.name}")
    return [
        _normalize_single_json_record(item, idx)
        for idx, item in enumerate(raw_items, start=1)
        if isinstance(item, dict) and (item.get("file_name") or item.get("File_Name") or item.get("filename"))
    ]


# Bersihkan folder input sementara sebelum memuat file baru
for old in INPUT_WORK_DIR.iterdir():
    if old.is_dir():
        shutil.rmtree(old)
    else:
        old.unlink()

nb1_source_paths = []
nb1_extracted_dir = None

if NB1_INPUT_SOURCE == "upload" and IN_COLAB:
    print("Silakan unggah file CSV / JSON / ZIP hasil Notebook 1 (bisa pilih 1 atau lebih file)...")
    uploaded_nb1 = colab_files.upload()
    for fname, content in uploaded_nb1.items():
        dst = INPUT_WORK_DIR / Path(fname).name
        dst.write_bytes(content)
        nb1_source_paths.append(dst)
elif NB1_INPUT_SOURCE == "drive" and IN_COLAB:
    ensure_drive_mounted()
    for cand in [DRIVE_NB1_INPUT_PATH, *DRIVE_NB1_FALLBACKS]:
        p = Path(cand)
        if p.is_file():
            nb1_source_paths.append(p)
            break
    if not nb1_source_paths:
        raise FileNotFoundError(f"File CSV/JSON Notebook 1 tidak ditemukan di Drive: {DRIVE_NB1_INPUT_PATH}")
else:
    p = Path(LOCAL_NB1_INPUT_PATH)
    if not p.is_file():
        raise FileNotFoundError(
            f"File input lokal tidak ditemukan: {p}. Ubah LOCAL_NB1_INPUT_PATH ke file CSV/JSON Notebook 1."
        )
    nb1_source_paths.append(p)

nb1_records = []
nb1_input_format_used = []

for src_path in nb1_source_paths:
    ext = src_path.suffix.lower()
    if ext == ".zip":
        nb1_extracted_dir = INPUT_WORK_DIR / "unzipped_nb1"
        nb1_extracted_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_path) as zf:
            zf.extractall(nb1_extracted_dir)
        json_all = next(nb1_extracted_dir.rglob("07_hasil_bounding_box_semua.json"), None)
        csv_all = next(nb1_extracted_dir.rglob("07_ringkasan_aturan_if_else.csv"), None)
        if json_all is not None:
            nb1_records.extend(parse_nb1_json(json_all))
            nb1_input_format_used.append(f"ZIP->{json_all.name}")
        elif csv_all is not None:
            nb1_records.extend(parse_nb1_csv(csv_all))
            nb1_input_format_used.append(f"ZIP->{csv_all.name}")
        else:
            per_jsons = sorted(nb1_extracted_dir.rglob("*.json"))
            for jf in per_jsons:
                try:
                    nb1_records.extend(parse_nb1_json(jf))
                except ValueError:
                    pass
            nb1_input_format_used.append(f"ZIP->({len(per_jsons)} JSON)")
    elif ext == ".json":
        nb1_records.extend(parse_nb1_json(src_path))
        nb1_input_format_used.append(f"JSON:{src_path.name}")
    elif ext == ".csv":
        nb1_records.extend(parse_nb1_csv(src_path))
        nb1_input_format_used.append(f"CSV:{src_path.name}")
    else:
        print(f"Melewati file non CSV/JSON/ZIP: {src_path.name}")

if not nb1_records:
    raise RuntimeError("Tidak ada entri gambar yang berhasil dibaca dari file CSV/JSON Notebook 1.")

# Pastikan urutan indeks rapi
for i, rec in enumerate(nb1_records, start=1):
    rec["index"] = i

has_bbox_count = sum(1 for r in nb1_records if r["nb1_bbox_xyxy_px"] is not None)
print(f"Sumber input Notebook 1 : {', '.join(nb1_input_format_used)}")
print(f"Total entri terbaca     : {len(nb1_records)} gambar")
print(f"Memiliki BBox NB1       : {has_bbox_count}/{len(nb1_records)} gambar")

df_nb1_preview = pd.DataFrame([
    {
        "No": r["index"],
        "File_Name": r["file_name"],
        "Ukuran": f"{r['image_width']}x{r['image_height']}",
        "Strategi_NB1": r["grabcut_strategy"],
        "NB1_BBox_xyxy": r["nb1_bbox_xyxy_px"],
        "Aspect_Ratio": r["nb1_aspect_ratio"],
    }
    for r in nb1_records[:10]
])
display(df_nb1_preview)

## Cell 4 — Hubungkan `File_Name` CSV/JSON ke Folder Gambar Asli

Karena file CSV/JSON dari Notebook 1 hanya berisi nama file (`File_Name`) dan koordinat BBox/fitur (bukan piksel gambar), MobileNet SSD perlu membaca citra aslinya berdasarkan `File_Name` tersebut.

Pilihan `IMAGE_SOURCE`:
* `"drive"` *(★ Default di Colab)*: Otomatis membaca gambar asli dari **`/content/drive/MyDrive/ferqueue/gambar`** (sama persis dengan folder sumber Notebook 1, termasuk pencarian rekursif di subfolder).
* `"upload"`: Unggah gambar uji (JPG/PNG) atau satu file ZIP berisi gambar-gambar uji.
* `"nb1_bbox_fallback"`: Hanya jika kamu mengunggah `nb1_pcv_klasik_outputs.zip` di Cell 3 dan tidak punya folder gambar asli *(memakai `per_gambar/*.bbox.jpg`, kurang ideal dibanding gambar bersih karena sudah ada garis hijau)*.
* `"local"`: Fallback saat dijalankan di laptop lokal FERQUEUE.

In [ ]:
IMAGE_SOURCE = "drive"  # Pilihan: "drive" | "upload" | "nb1_bbox_fallback" | "local"
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/ferqueue/gambar"
LOCAL_IMAGE_DIRS = ["data/helper", "outputs/test-samples", "data/inference", "data/raw/vehicles"]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
UPLOADED_IMG_DIR = OUT_DIR / "_uploaded_images"


def build_image_index(dirs_to_scan: list) -> tuple:
    by_exact = {}
    by_lower = {}
    by_stem = {}
    total_found = 0
    for d in dirs_to_scan:
        root = Path(d)
        if not root.is_dir():
            continue
        for p in sorted(root.rglob("*")):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                total_found += 1
                by_exact.setdefault(p.name, p)
                by_lower.setdefault(p.name.lower(), p)
                by_stem.setdefault(p.stem.lower(), p)
    return by_exact, by_lower, by_stem, total_found


search_dirs = []
if IMAGE_SOURCE == "drive" and IN_COLAB:
    ensure_drive_mounted()
    src_dir = Path(DRIVE_IMAGE_DIR)
    if not src_dir.is_dir():
        raise FileNotFoundError(
            f"Folder gambar Google Drive tidak ditemukan: {src_dir}. "
            "Ubah DRIVE_IMAGE_DIR atau ganti IMAGE_SOURCE = 'upload'."
        )
    search_dirs.append(src_dir)
elif IMAGE_SOURCE == "upload" and IN_COLAB:
    if UPLOADED_IMG_DIR.exists():
        shutil.rmtree(UPLOADED_IMG_DIR)
    UPLOADED_IMG_DIR.mkdir(parents=True, exist_ok=True)
    print("Silakan unggah gambar uji (JPG/PNG) atau 1 file ZIP yang berisi gambar uji...")
    up_imgs = colab_files.upload()
    for fname, content in up_imgs.items():
        dst = UPLOADED_IMG_DIR / Path(fname).name
        dst.write_bytes(content)
        if dst.suffix.lower() == ".zip":
            with zipfile.ZipFile(dst) as zf:
                zf.extractall(UPLOADED_IMG_DIR / dst.stem)
    search_dirs.append(UPLOADED_IMG_DIR)
elif IMAGE_SOURCE == "nb1_bbox_fallback":
    if nb1_extracted_dir is None:
        raise RuntimeError("Mode 'nb1_bbox_fallback' memerlukan upload nb1_pcv_klasik_outputs.zip pada Cell 3.")
    search_dirs.append(nb1_extracted_dir)
else:
    search_dirs.extend(Path(d) for d in LOCAL_IMAGE_DIRS if Path(d).is_dir())
    if nb1_extracted_dir is not None:
        search_dirs.append(nb1_extracted_dir)

by_exact, by_lower, by_stem, total_disk_imgs = build_image_index(search_dirs)

matched_records = []
missing_records = []
for rec in nb1_records:
    fname = rec["file_name"]
    stem = Path(fname).stem.lower()
    img_path = by_exact.get(fname) or by_lower.get(fname.lower()) or by_stem.get(stem)

    # Fallback tambahan ke per_gambar/*.bbox.jpg jika upload ZIP NB1 dan gambar asli tidak ditemukan
    if img_path is None and nb1_extracted_dir is not None and rec.get("nb1_bbox_image_file"):
        cand_bbox = nb1_extracted_dir / rec["nb1_bbox_image_file"]
        if cand_bbox.is_file():
            img_path = cand_bbox

    if img_path is not None:
        rec["image_path"] = img_path
        matched_records.append(rec)
    else:
        missing_records.append(fname)

print(f"Folder gambar dipindai  : {[str(d) for d in search_dirs]} ({total_disk_imgs} file gambar)")
print(f"Cocok dengan CSV/JSON   : {len(matched_records)}/{len(nb1_records)} gambar")
if missing_records:
    print(f"PERINGATAN: {len(missing_records)} file di CSV/JSON tidak ditemukan gambarnya. Contoh: {missing_records[:5]}")

if not matched_records:
    raise FileNotFoundError(
        "Tidak ada gambar yang cocok dengan File_Name di CSV/JSON! "
        "Pastikan DRIVE_IMAGE_DIR mengarah ke folder gambar asli yang sama dengan Notebook 1, "
        "atau ganti IMAGE_SOURCE = 'upload'."
    )

## Cell 5 — Konfigurasi Inferensi SSD MobileNet V2 & Helper Integrasi BBox Notebook 1

Atur parameter di bawah sesuai kebutuhan:
* **`INFERENCE_MODE = "full_image"`** *(★ Default / Rekomendasi)*: SSD membaca gambar utuh sesuai kondisi training (akurasi terbaik), lalu membandingkan IoU dengan BBox Notebook 1 dan menuliskan jenis kendaraan + BBox SSD ke output.
* **`INFERENCE_MODE = "roi_crop"`**: SSD membaca potongan area BBox dari CSV/JSON Notebook 1 (+ margin `ROI_PADDING_RATIO = 0.15`), lalu memetakan koordinat BBox hasil deteksi SSD kembali ke koordinat gambar asli. Jika di dalam crop ROI tidak terdeteksi dan `FALLBACK_TO_FULL_IMAGE = True`, otomatis melakukan inferensi ulang pada gambar utuh.
* **`CONFIDENCE_THRESHOLD = 0.50`**: sama dengan `receiver-v4.json` (turunkan ke `0.35` jika ingin lebih sensitif).
* **`MAX_IMAGES_TO_PROCESS = 0`**: `0` = proses **semua** gambar di CSV/JSON; isi angka (misal `50`) jika ingin uji cepat sebagian gambar dulu.

In [ ]:
INFERENCE_MODE = "full_image"   # Pilihan: "full_image" (rekomendasi) | "roi_crop"
ROI_PADDING_RATIO = 0.15        # Padding 15% di sekitar BBox NB1 saat INFERENCE_MODE = "roi_crop"
FALLBACK_TO_FULL_IMAGE = True   # Jika roi_crop tidak mendeteksi kendaraan, otomatis cek full image

CONFIDENCE_THRESHOLD = 0.50     # Sesuai apps/computer-vision/configs/receiver-v4.json
NMS_IOU_THRESHOLD = 0.45
MAX_DETECTIONS = 10
CLASS_AGNOSTIC_NMS = True

MAX_IMAGES_TO_PROCESS = 0       # 0 = proses SEMUA gambar di CSV/JSON | >0 = batasi untuk uji cepat
DISPLAY_SAMPLE_COUNT = 6        # Jumlah sampel visual yang ditampilkan di Colab
SAMPLE_SEED = 42                # None = acak tiap run | angka = selalu sama
DRAW_NB1_REF_BOX = True         # Gambar kotak referensi putus-putus kuning dari BBox NB1 di foto output
SAVE_PER_IMAGE_OUTPUTS = True   # Simpan per_gambar/*.ssd.jpg & per_gambar/*.ssd.json

CLASS_COLORS = {
    "Mobil": (40, 210, 90),
    "Truk": (255, 165, 0),
    "Bus": (65, 155, 255),
}
NB1_REF_COLOR = (255, 225, 40)


def load_pil_rgb(path: Path) -> Image.Image:
    with Image.open(path) as img:
        return ImageOps.exif_transpose(img).convert("RGB")


def preprocess_image_ferqueue(image: Image.Image, settings=None) -> Image.Image:
    '''Persis mengikuti apps/computer-vision/src/ferqueue_cv/processing.py.'''
    settings = settings or {}
    image = image.convert("RGB")
    kernel = settings.get("denoise_kernel", 0)
    clip = settings.get("clahe_clip_limit", 0)
    gamma = settings.get("gamma", 1.0)
    if not kernel and not clip and gamma == 1.0:
        return image
    pixels = np.asarray(image).copy()
    if kernel:
        pixels = cv2.GaussianBlur(pixels, (kernel, kernel), 0)
    if clip:
        lab = cv2.cvtColor(pixels, cv2.COLOR_RGB2LAB)
        grid = settings.get("clahe_grid_size", 8)
        lab[:, :, 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(lab[:, :, 0])
        pixels = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    if gamma != 1.0:
        table = np.round(255 * (np.arange(256) / 255) ** gamma).astype("uint8")
        pixels = cv2.LUT(pixels, table)
    return Image.fromarray(pixels)


def select_detections_ferqueue(result: dict, conf_th=0.5, iou_th=0.45, max_det=10, class_agnostic=True):
    '''Persis mengikuti apps/computer-vision/src/ferqueue_cv/postprocessing.py.'''
    boxes = np.asarray(result["detection_boxes"])
    scores = np.asarray(result["detection_scores"])
    classes = np.asarray(result["detection_classes"])
    candidates = [
        int(idx)
        for idx in np.argsort(-scores, kind="stable")
        if scores[idx] >= conf_th
        and np.isfinite(scores[idx])
        and np.isfinite(boxes[idx]).all()
        and np.isfinite(classes[idx])
        and boxes[idx][2] > boxes[idx][0]
        and boxes[idx][3] > boxes[idx][1]
    ]
    keep = []
    while candidates and len(keep) < max_det:
        current = candidates.pop(0)
        keep.append(current)
        remaining = []
        a = boxes[current]
        for idx in candidates:
            if not class_agnostic and classes[idx] != classes[current]:
                remaining.append(idx)
                continue
            b = boxes[idx]
            inter = max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
            union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
            if union <= 0 or inter / union <= iou_th:
                remaining.append(idx)
        candidates = remaining
    return keep


def compute_iou_xyxy(box_a, box_b) -> float:
    if not box_a or not box_b:
        return 0.0
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return round(inter / union, 4) if union > 0 else 0.0


def run_ssd_on_rgb(rgb: np.ndarray, offset_xy=(0, 0), full_wh=None):
    img_h, img_w = rgb.shape[:2]
    full_w, full_h = full_wh if full_wh is not None else (img_w, img_h)
    ox, oy = offset_xy

    t0 = perf_counter()
    raw = detector_fn(tf.convert_to_tensor(rgb[None, ...], dtype=tf.uint8))
    elapsed_ms = (perf_counter() - t0) * 1000.0

    count = int(raw["num_detections"][0])
    parsed = {
        "detection_boxes": raw["detection_boxes"][0, :count].numpy(),
        "detection_scores": raw["detection_scores"][0, :count].numpy(),
        "detection_classes": raw["detection_classes"][0, :count].numpy(),
    }
    kept = select_detections_ferqueue(
        parsed, CONFIDENCE_THRESHOLD, NMS_IOU_THRESHOLD, MAX_DETECTIONS, CLASS_AGNOSTIC_NMS
    )

    detections = []
    for det_rank, idx in enumerate(kept, start=1):
        ny1, nx1, ny2, nx2 = map(float, parsed["detection_boxes"][idx])
        score = float(parsed["detection_scores"][idx])
        cid = int(parsed["detection_classes"][idx])
        cname = CLASSES[cid - 1] if 1 <= cid <= len(CLASSES) else f"ID-{cid}"

        px1 = max(0, min(full_w, int(round(ox + nx1 * img_w))))
        py1 = max(0, min(full_h, int(round(oy + ny1 * img_h))))
        px2 = max(0, min(full_w, int(round(ox + nx2 * img_w))))
        py2 = max(0, min(full_h, int(round(oy + ny2 * img_h))))
        if px2 <= px1 or py2 <= py1:
            continue

        detections.append({
            "detection_id": det_rank,
            "class_id": cid,
            "class_name": cname,
            "confidence": round(score, 4),
            "bbox_xyxy_px": [px1, py1, px2, py2],
            "bbox_xywh_px": [px1, py1, px2 - px1, py2 - py1],
            "bbox_norm_xyxy": [
                round(px1 / float(full_w), 6),
                round(py1 / float(full_h), 6),
                round(px2 / float(full_w), 6),
                round(py2 / float(full_h), 6),
            ],
            "bbox_norm_yxyx": [
                round(py1 / float(full_h), 6),
                round(px1 / float(full_w), 6),
                round(py2 / float(full_h), 6),
                round(px2 / float(full_w), 6),
            ],
        })
    return detections, elapsed_ms


def detect_from_nb1_record(rgb: np.ndarray, nb1_bbox_xyxy):
    full_h, full_w = rgb.shape[:2]
    if INFERENCE_MODE == "roi_crop" and nb1_bbox_xyxy is not None:
        x1, y1, x2, y2 = nb1_bbox_xyxy
        bw, bh = max(1, x2 - x1), max(1, y2 - y1)
        pad_x = int(round(bw * ROI_PADDING_RATIO))
        pad_y = int(round(bh * ROI_PADDING_RATIO))
        cx1 = max(0, x1 - pad_x)
        cy1 = max(0, y1 - pad_y)
        cx2 = min(full_w, x2 + pad_x)
        cy2 = min(full_h, y2 + pad_y)
        if cx2 - cx1 >= 16 and cy2 - cy1 >= 16:
            roi = rgb[cy1:cy2, cx1:cx2]
            dets, ms = run_ssd_on_rgb(roi, offset_xy=(cx1, cy1), full_wh=(full_w, full_h))
            if dets or not FALLBACK_TO_FULL_IMAGE:
                return dets, ms, "roi_crop_nb1"
            dets_full, ms_full = run_ssd_on_rgb(rgb, offset_xy=(0, 0), full_wh=(full_w, full_h))
            return dets_full, ms + ms_full, "fallback_full_image"
    dets, ms = run_ssd_on_rgb(rgb, offset_xy=(0, 0), full_wh=(full_w, full_h))
    return dets, ms, "full_image"


def draw_dashed_rect(img: np.ndarray, pt1, pt2, color, thickness=2, dash=12):
    x1, y1 = pt1
    x2, y2 = pt2
    for x in range(x1, x2, dash * 2):
        cv2.line(img, (x, y1), (min(x + dash, x2), y1), color, thickness, cv2.LINE_AA)
        cv2.line(img, (x, y2), (min(x + dash, x2), y2), color, thickness, cv2.LINE_AA)
    for y in range(y1, y2, dash * 2):
        cv2.line(img, (x1, y), (x1, min(y + dash, y2)), color, thickness, cv2.LINE_AA)
        cv2.line(img, (x2, y), (x2, min(y + dash, y2)), color, thickness, cv2.LINE_AA)


def render_ssd_overlay(rgb: np.ndarray, detections: list, nb1_bbox_xyxy=None, draw_nb1=True) -> np.ndarray:
    vis = rgb.copy()
    h, w = vis.shape[:2]
    thick = max(2, int(round(min(h, w) / 220.0)))
    font_scale = max(0.55, min(1.0, min(h, w) / 600.0))

    if draw_nb1 and nb1_bbox_xyxy is not None:
        nx1, ny1, nx2, ny2 = nb1_bbox_xyxy
        draw_dashed_rect(vis, (nx1, ny1), (nx2, ny2), NB1_REF_COLOR, thickness=max(2, thick - 1))

    for det in detections:
        x1, y1, x2, y2 = det["bbox_xyxy_px"]
        cname = det["class_name"]
        conf = det["confidence"]
        color = CLASS_COLORS.get(cname, (40, 210, 90))
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, thick)
        label_txt = f"{cname} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 2)
        top_y = max(0, y1 - th - 10)
        cv2.rectangle(vis, (x1, top_y), (min(w - 1, x1 + tw + 10), top_y + th + 10), color, -1)
        cv2.putText(
            vis, label_txt, (x1 + 5, top_y + th + 5),
            cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), 2, cv2.LINE_AA,
        )
    return vis


# Warmup 1x agar catatan waktu inferensi akurat
_ = detector_fn(tf.zeros([1, 320, 320, 3], dtype=tf.uint8))
print(
    f"Konfigurasi siap! Mode={INFERENCE_MODE} | Conf>={CONFIDENCE_THRESHOLD} | "
    f"NMS IoU<={NMS_IOU_THRESHOLD} | Model={model_meta['checkpoint']}"
)

## Cell 6 — Jalankan Deteksi MobileNet SSD pada Seluruh Input CSV/JSON Notebook 1

Memproses gambar secara **streaming 1 per 1** agar tetap hemat RAM meski jumlah baris CSV/JSON mencapai ratusan–ribuan gambar, lalu menyimpan hasil deteksi (jenis kendaraan, confidence, bounding box SSD, serta perbandingan IoU terhadap BBox Notebook 1).

In [ ]:
PER_IMAGE_DIR = OUT_DIR / "per_gambar"
if PER_IMAGE_DIR.exists():
    shutil.rmtree(PER_IMAGE_DIR)
if SAVE_PER_IMAGE_OUTPUTS:
    PER_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

work_records = matched_records[:MAX_IMAGES_TO_PROCESS] if MAX_IMAGES_TO_PROCESS > 0 else list(matched_records)
total_imgs = len(work_records)

rng = random.Random(SAMPLE_SEED)
sample_k = min(DISPLAY_SAMPLE_COUNT, total_imgs)
sample_indices = set(rng.sample(range(total_imgs), sample_k)) if total_imgs > sample_k else set(range(total_imgs))

csv_rows = []
json_results = []
sample_vis_items = []
failed_items = []

t_start = perf_counter()
print(f"Memulai deteksi SSD MobileNet V2 pada {total_imgs} gambar dari CSV/JSON Notebook 1...")

for idx, rec in enumerate(work_records):
    img_path = rec["image_path"]
    name = rec["file_name"]
    stem = Path(name).stem
    try:
        pil_img = load_pil_rgb(img_path)
        proc_pil = preprocess_image_ferqueue(pil_img, MODEL_PREPROCESSING)
        rgb = np.asarray(proc_pil, dtype=np.uint8)
        img_h, img_w = rgb.shape[:2]

        nb1_box = rec["nb1_bbox_xyxy_px"]
        if nb1_box is not None:
            nb1_box = [
                max(0, min(img_w, int(nb1_box[0]))),
                max(0, min(img_h, int(nb1_box[1]))),
                max(0, min(img_w, int(nb1_box[2]))),
                max(0, min(img_h, int(nb1_box[3]))),
            ]
            if nb1_box[2] <= nb1_box[0] or nb1_box[3] <= nb1_box[1]:
                nb1_box = None

        detections, infer_ms, infer_src = detect_from_nb1_record(rgb, nb1_box)
        for det in detections:
            det["iou_vs_nb1_bbox"] = compute_iou_xyxy(det["bbox_xyxy_px"], nb1_box) if nb1_box else None

        if detections:
            primary = detections[0]
            primary_label = primary["class_name"]
            primary_conf = primary["confidence"]
            sx1, sy1, sx2, sy2 = primary["bbox_xyxy_px"]
            sw, sh = sx2 - sx1, sy2 - sy1
            snx1, sny1, snx2, sny2 = primary["bbox_norm_xyxy"]
            best_iou = max((d["iou_vs_nb1_bbox"] or 0.0) for d in detections) if nb1_box else None
            all_labels_str = ", ".join(f"{d['class_name']} ({d['confidence']:.2f})" for d in detections)
        else:
            primary_label = "Tidak Terdeteksi"
            primary_conf = 0.0
            sx1 = sy1 = sx2 = sy2 = sw = sh = 0
            snx1 = sny1 = snx2 = sny2 = 0.0
            best_iou = 0.0 if nb1_box else None
            all_labels_str = ""

        out_img_rel = ""
        out_json_rel = ""
        vis_ssd_only = None
        vis_combined = None

        if SAVE_PER_IMAGE_OUTPUTS or idx in sample_indices:
            vis_ssd_only = render_ssd_overlay(rgb, detections, nb1_bbox_xyxy=None, draw_nb1=False)
            vis_combined = render_ssd_overlay(rgb, detections, nb1_bbox_xyxy=nb1_box, draw_nb1=DRAW_NB1_REF_BOX)

        if SAVE_PER_IMAGE_OUTPUTS:
            out_img_name = f"{idx + 1:04d}_{stem}.ssd.jpg"
            out_json_name = f"{idx + 1:04d}_{stem}.ssd.json"
            out_img_rel = f"per_gambar/{out_img_name}"
            out_json_rel = f"per_gambar/{out_json_name}"
            Image.fromarray(vis_combined).save(PER_IMAGE_DIR / out_img_name, format="JPEG", quality=92)

        record_out = {
            "index": idx + 1,
            "file_name": name,
            "image_width": img_w,
            "image_height": img_h,
            "inference_source": infer_src,
            "inference_ms": round(infer_ms, 2),
            "nb1_input": {
                "grabcut_strategy": rec["grabcut_strategy"],
                "bbox_xyxy_px": nb1_box,
                "aspect_ratio": rec["nb1_aspect_ratio"],
                "extent": rec["nb1_extent"],
                "solidity": rec["nb1_solidity"],
                "circularity": rec["nb1_circularity"],
                "mask_coverage_pct": rec["nb1_mask_coverage_pct"],
            },
            "primary_label": primary_label,
            "primary_confidence": primary_conf,
            "detection_count": len(detections),
            "iou_nb1_vs_ssd": best_iou,
            "ssd_detections": detections,
            "output_ssd_image": out_img_rel,
        }
        if SAVE_PER_IMAGE_OUTPUTS:
            (PER_IMAGE_DIR / out_json_name).write_text(
                json.dumps(record_out, indent=2, ensure_ascii=False), encoding="utf-8"
            )
        json_results.append(record_out)

        nb1_x1, nb1_y1, nb1_x2, nb1_y2 = nb1_box if nb1_box else (0, 0, 0, 0)
        csv_rows.append({
            "No": idx + 1,
            "File_Name": name,
            "Image_W": img_w,
            "Image_H": img_h,
            "Label": primary_label,
            "SSD_Confidence": primary_conf,
            "SSD_Jumlah_Deteksi": len(detections),
            "SSD_Semua_Deteksi": all_labels_str,
            "SSD_BBox_xmin": sx1,
            "SSD_BBox_ymin": sy1,
            "SSD_BBox_xmax": sx2,
            "SSD_BBox_ymax": sy2,
            "SSD_BBox_w": sw,
            "SSD_BBox_h": sh,
            "SSD_Norm_xmin": snx1,
            "SSD_Norm_ymin": sny1,
            "SSD_Norm_xmax": snx2,
            "SSD_Norm_ymax": sny2,
            "NB1_Strategi": rec["grabcut_strategy"],
            "NB1_BBox_xmin": nb1_x1,
            "NB1_BBox_ymin": nb1_y1,
            "NB1_BBox_xmax": nb1_x2,
            "NB1_BBox_ymax": nb1_y2,
            "IoU_NB1_vs_SSD": best_iou if best_iou is not None else "",
            "Sumber_Inferensi": infer_src,
            "Inferensi_ms": round(infer_ms, 2),
            "Output_SSD_Image": out_img_rel,
            "Output_SSD_JSON": out_json_rel,
        })

        if idx in sample_indices:
            vis_nb1 = rgb.copy()
            if nb1_box is not None:
                cv2.rectangle(vis_nb1, (nb1_box[0], nb1_box[1]), (nb1_box[2], nb1_box[3]), (40, 220, 80), 4)
            sample_vis_items.append({
                "name": name,
                "vis_nb1": vis_nb1,
                "vis_ssd": vis_ssd_only,
                "vis_combined": vis_combined,
                "primary_label": primary_label,
                "primary_conf": primary_conf,
                "det_count": len(detections),
                "iou": best_iou,
                "infer_ms": infer_ms,
                "infer_src": infer_src,
            })
    except Exception as exc:
        failed_items.append({"file_name": name, "error": str(exc)})

    done = idx + 1
    log_every = 25 if total_imgs >= 50 else max(1, total_imgs // 4)
    if done % log_every == 0 or done == total_imgs:
        elapsed = perf_counter() - t_start
        rate = done / max(elapsed, 1e-6)
        eta = (total_imgs - done) / max(rate, 1e-6)
        print(f"  Progres: {done}/{total_imgs} ({100.0 * done / total_imgs:.1f}%) | {elapsed:.1f} dtk | sisa ~{eta:.0f} dtk")

df_results = pd.DataFrame(csv_rows)
print(f"\nSelesai! Berhasil diproses: {len(df_results)} | Gagal: {len(failed_items)}")
display(
    df_results.head(15)[[
        "No", "File_Name", "Label", "SSD_Confidence", "SSD_Jumlah_Deteksi",
        "SSD_BBox_xmin", "SSD_BBox_ymin", "SSD_BBox_w", "SSD_BBox_h", "IoU_NB1_vs_SSD", "Inferensi_ms",
    ]]
)

## Cell 7 — Visualisasi Perbandingan (BBox Notebook 1 vs Deteksi SSD MobileNet V2) & Statistik

Menampilkan:
1. **Perbandingan visual 3 kolom** pada sampel gambar:
   - Kolom 1: **Input BBox PCV Klasik (Notebook 1, tanpa label)**
   - Kolom 2: **Deteksi SSD MobileNet V2 (Jenis Kendaraan + Confidence + BBox)**
   - Kolom 3: **Overlay Gabungan** (garis kuning putus-putus = BBox NB1, kotak berwarna solid = BBox SSD + nilai `IoU`)
2. **Grafik statistik**: distribusi kelas terdeteksi (`Mobil`, `Truk`, `Bus`, `Tidak Terdeteksi`), sebaran confidence score, dan distribusi IoU terhadap BBox Notebook 1.

In [ ]:
# 1. Visualisasi 3-kolom pada sampel gambar
if sample_vis_items:
    n_rows = len(sample_vis_items)
    fig, axes = plt.subplots(n_rows, 3, figsize=(15.5, 4.0 * n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, item in enumerate(sample_vis_items):
        iou_str = f"IoU={item['iou']:.2f}" if item["iou"] is not None else "IoU=N/A"
        axes[r, 0].imshow(item["vis_nb1"])
        axes[r, 0].set_title(f"1. Input NB1 (Tanpa Label)\n{item['name'][:28]}", fontsize=9.5)
        axes[r, 1].imshow(item["vis_ssd"])
        axes[r, 1].set_title(
            f"2. SSD MobileNet V2\n{item['primary_label']} ({item['primary_conf']:.2f}) | {item['infer_ms']:.0f} ms",
            fontsize=9.5,
        )
        axes[r, 2].imshow(item["vis_combined"])
        axes[r, 2].set_title(f"3. Overlay NB1 (Kuning) vs SSD\n{iou_str} | mode={item['infer_src']}", fontsize=9.5)
        for c in range(3):
            axes[r, c].axis("off")

    fig.suptitle(
        f"Hasil Deteksi SSD MobileNet V2 ({model_meta['checkpoint']}) dari Input CSV/JSON Notebook 1",
        fontsize=13.5, fontweight="bold",
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    save_fig(fig, "01_perbandingan_nb1_vs_ssd.png")
    plt.show()

# 2. Ringkasan statistik & grafik distribusi
if len(df_results) > 0:
    detected_mask = df_results["SSD_Jumlah_Deteksi"] > 0
    det_rate = 100.0 * float(detected_mask.mean())
    mean_ms = float(df_results["Inferensi_ms"].mean())
    confs = df_results.loc[detected_mask, "SSD_Confidence"].astype(float)
    ious = pd.to_numeric(df_results["IoU_NB1_vs_SSD"], errors="coerce").dropna()

    summary_table = pd.DataFrame([
        {"Metrik": "Total Gambar Diproses", "Nilai": len(df_results)},
        {"Metrik": "Kendaraan Terdeteksi", "Nilai": f"{int(detected_mask.sum())} ({det_rate:.1f}%)"},
        {"Metrik": "Tidak Terdeteksi", "Nilai": int((~detected_mask).sum())},
        {"Metrik": "Rata-rata Confidence (terdeteksi)", "Nilai": f"{confs.mean():.4f}" if len(confs) else "-"},
        {"Metrik": "Rata-rata IoU (BBox NB1 vs SSD)", "Nilai": f"{ious.mean():.4f}" if len(ious) else "-"},
        {"Metrik": "Rata-rata Waktu Inferensi", "Nilai": f"{mean_ms:.1f} ms/gambar"},
    ])
    display(summary_table)

    fig2, axes2 = plt.subplots(1, 3, figsize=(16, 4.4))

    label_order = [*CLASSES, "Tidak Terdeteksi"]
    counts = [int((df_results["Label"] == lbl).sum()) for lbl in label_order]
    bar_colors = ["#28d25a", "#ffa500", "#419bff", "#888888"][: len(label_order)]
    bars = axes2[0].bar(label_order, counts, color=bar_colors)
    for b, v in zip(bars, counts):
        axes2[0].text(b.get_x() + b.get_width() / 2, v + max(1, max(counts) * 0.02), str(v), ha="center", fontweight="bold")
    axes2[0].set_title("Distribusi Jenis Kendaraan Terdeteksi")
    axes2[0].set_ylabel("Jumlah Gambar")
    axes2[0].grid(axis="y", alpha=0.25)

    if len(confs):
        axes2[1].hist(confs, bins=20, range=(CONFIDENCE_THRESHOLD, 1.0), color="#4c72b0", alpha=0.85, edgecolor="white")
    axes2[1].set_title("Sebaran Confidence Score SSD")
    axes2[1].set_xlabel("Confidence")
    axes2[1].set_ylabel("Frekuensi")
    axes2[1].grid(alpha=0.25)

    if len(ious):
        axes2[2].hist(ious, bins=20, range=(0.0, 1.0), color="#dd8452", alpha=0.85, edgecolor="white")
        axes2[2].axvline(ious.mean(), color="red", linestyle="--", label=f"Mean={ious.mean():.2f}")
        axes2[2].legend()
    axes2[2].set_title("Distribusi IoU: BBox NB1 vs BBox SSD")
    axes2[2].set_xlabel("IoU (0..1)")
    axes2[2].set_ylabel("Frekuensi")
    axes2[2].grid(alpha=0.25)

    fig2.suptitle("Ringkasan Statistik Deteksi SSD MobileNet V2", fontsize=13.5, fontweight="bold")
    plt.tight_layout()
    save_fig(fig2, "02_distribusi_kelas_dan_statistik.png")
    plt.show()

## Cell 8 — Ekspor Hasil Akhir (CSV Ter-Label, JSON Gabungan, Gambar Ber-BBox, & ZIP Siap Unduh)

Menyimpan semua hasil deteksi ke:
1. **`hasil_deteksi_ssd_dari_nb1.csv`** — tabel lengkap berisi jenis kendaraan (`Label`), confidence, koordinat BBox SSD, koordinat BBox NB1, dan IoU.
2. **`hasil_deteksi_ssd_dari_nb1.json`** — JSON gabungan terstruktur untuk seluruh gambar.
3. **`per_gambar/`** — gambar hasil deteksi ber-BBox + label & JSON per gambar.
4. **`nb5_ssd_dari_nb1_outputs.zip`** — paket ZIP lengkap (otomatis disalin ke Google Drive `MyDrive/ferqueue/output_ssd_dari_nb1/` jika Drive ter-mount).

In [ ]:
out_csv_path = OUT_DIR / "hasil_deteksi_ssd_dari_nb1.csv"
out_json_path = OUT_DIR / "hasil_deteksi_ssd_dari_nb1.json"

df_results.to_csv(out_csv_path, index=False)

summary_payload = {
    "pipeline": "ssd_mobilenet_v2_from_nb1_input",
    "model": model_meta,
    "inference_config": {
        "inference_mode": INFERENCE_MODE,
        "roi_padding_ratio": ROI_PADDING_RATIO,
        "fallback_to_full_image": FALLBACK_TO_FULL_IMAGE,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "max_detections": MAX_DETECTIONS,
        "class_agnostic_nms": CLASS_AGNOSTIC_NMS,
    },
    "nb1_sources": nb1_input_format_used,
    "total_processed": len(json_results),
    "failed_count": len(failed_items),
    "failed_items": failed_items,
    "class_counts": df_results["Label"].value_counts().to_dict() if len(df_results) else {},
    "results": json_results,
}
out_json_path.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")

zip_out_path = OUT_DIR.parent / "nb5_ssd_dari_nb1_outputs.zip"
with zipfile.ZipFile(zip_out_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.rglob("*")):
        rel = p.relative_to(OUT_DIR)
        if p.is_file() and not any(part.startswith("_") for part in rel.parts):
            zf.write(p, rel.as_posix())

print(f"CSV hasil deteksi  : {out_csv_path}")
print(f"JSON hasil deteksi : {out_json_path}")
print(f"Arsip ZIP lengkap  : {zip_out_path} ({zip_out_path.stat().st_size / (1024**2):.2f} MB)")

# Salin otomatis ke Google Drive jika ter-mount
drive_out_dir = Path("/content/drive/MyDrive/ferqueue/output_ssd_dari_nb1")
if IN_COLAB and Path("/content/drive/MyDrive").is_dir():
    drive_out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(out_csv_path, drive_out_dir / out_csv_path.name)
    shutil.copy2(out_json_path, drive_out_dir / out_json_path.name)
    shutil.copy2(zip_out_path, drive_out_dir / zip_out_path.name)
    print(f"Salinan CSV, JSON, & ZIP tersimpan di Google Drive: {drive_out_dir}")

# Hapus tanda '#' di bawah jika ingin langsung mengunduh CSV/JSON/ZIP lewat browser Colab:
# if IN_COLAB:
#     colab_files.download(str(out_csv_path))
#     colab_files.download(str(out_json_path))
#     colab_files.download(str(zip_out_path))